# LLM Performance Analysis: CPU vs GPU Comparative Study

## Objective

Determine which platform (CPU or GPU) is better for running Large Language Models and quantify the performance difference.

### Key Questions:
1. **Which is faster?** CPU or GPU for LLM inference?
2. **By how much?** What is the quantitative speedup factor?
3. **Does it depend on workload?** How do model size and prompt length affect performance?
4. **Is it consistent?** How much variability exists in the measurements?
5. **What's the verdict?** Clear recommendation for deployment

### Methodology:
- **Iterative analysis** - Results analyzed step by step
- **Data-driven** - Decisions based on actual measurements
- **Comprehensive** - Multiple models, prompt lengths, and replications
- **Visual** - Clear graphs and heatmaps for insights


## Configuration

Set up the analysis parameters. Update `TEST_NAME` and `MACHINE_ID` for each new dataset.


In [ ]:
# === CONFIGURATION ===
TEST_NAME = "Test8"  # Update for each test: Test1, Test2, Test3, etc.
DATA_PATH = f"../Results/{TEST_NAME}/results.csv"
MACHINE_ID = "Machine_A"  # Update for different servers
OUTPUT_DIR = f"../Results/{TEST_NAME}/plots_{MACHINE_ID}"

print(f"📊 Analysis Configuration")
print(f"{'='*50}")
print(f"Test Dataset:    {TEST_NAME}")
print(f"Machine ID:      {MACHINE_ID}")
print(f"Data Source:     {DATA_PATH}")
print(f"Output Folder:   {OUTPUT_DIR}")
print(f"{'='*50}")


## Step 1: Data Loading & Initial Validation

Load the data and perform basic validation checks.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import linregress, ttest_ind
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import warnings
import os
warnings.filterwarnings('ignore')

# Plotting configuration
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✓ Output directory ready: {OUTPUT_DIR}\n")

# Load data
try:
    df = pd.read_csv(DATA_PATH)
    print(f"✓ Data loaded successfully")
    print(f"  → {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"  → Columns: {list(df.columns)}")
except FileNotFoundError:
    print(f"❌ ERROR: File not found at {DATA_PATH}")
    print("Please check the path and try again.")
    df = None

if df is not None:
    print(f"\n📋 First few rows:")
    display(df.head())


### Validation Checks


In [ ]:
if df is not None:
    print("🔍 DATA VALIDATION")
    print(f"{'='*50}\n")
    
    # Missing values
    missing = df.isnull().sum()
    if missing.sum() == 0:
        print("✓ No missing values")
    else:
        print("⚠ Missing values detected:")
        print(missing[missing > 0])
    
    # Data ranges
    print(f"\n📊 Data Ranges:")
    print(f"  Models:         {list(df['model'].unique())}")
    print(f"  Backends:       {list(df['backend'].unique())}")
    print(f"  Prompt lengths: {df['words'].min()} to {df['words'].max()} words")
    print(f"  Run numbers:    {sorted(df['run'].unique())}")
    print(f"  Time range:     {df['seconds'].min():.3f}s to {df['seconds'].max():.3f}s")
    
    # Dataset balance
    print(f"\n⚖️  Dataset Balance:")
    print(df.groupby(['model', 'backend']).size())


## Step 2: Data Preprocessing

Extract model sizes and normalize data for analysis.


In [ ]:
if df is not None:
    # Extract model size
    def get_model_family(model_name):
        for tag in ["3b", "8b", "13b", "70b"]:
            if tag in model_name.lower():
                return tag
        return "other"
    
    df["model_size"] = df["model"].apply(get_model_family)
    df["backend"] = df["backend"].str.upper()
    df["machine"] = MACHINE_ID
    
    print("✓ Data preprocessing completed")
    print(f"\n  Model sizes found: {list(df['model_size'].unique())}")
    print(f"  Backends:          {list(df['backend'].unique())}")
    
    # Sample of preprocessed data
    print(f"\n📋 Preprocessed Sample:")
    display(df[["model", "model_size", "backend", "words", "run", "seconds"]].head(10))


## Step 3: Descriptive Statistics

### Overall Summary


In [ ]:
if df is not None:
    print("📈 OVERALL STATISTICS")
    print(f"{'='*50}\n")
    print(df["seconds"].describe())
    print(f"\nTotal measurements: {len(df)}")
    print(f"Unique configurations: {len(df.groupby(['model', 'backend', 'words']))}")


### Statistics by Backend


In [ ]:
if df is not None:
    print("🖥️  CPU vs 🎮 GPU COMPARISON")
    print(f"{'='*50}\n")
    
    backend_stats = df.groupby("backend")["seconds"].describe()
    display(backend_stats)
    
    # Quick comparison
    cpu_mean = df[df.backend == "CPU"]["seconds"].mean()
    gpu_mean = df[df.backend == "GPU"]["seconds"].mean()
    
    print(f"\n⚡ Quick Comparison:")
    print(f"  CPU Average: {cpu_mean:.3f}s")
    print(f"  GPU Average: {gpu_mean:.3f}s")
    print(f"  Ratio: GPU is {cpu_mean/gpu_mean:.2f}x faster")


### Statistics by Model Size


In [ ]:
if df is not None:
    print("📦 STATISTICS BY MODEL SIZE")
    print(f"{'='*50}\n")
    display(df.groupby("model_size")["seconds"].describe())


## Step 4: Aggregated Statistics

Create summary statistics grouped by configuration.


In [ ]:
if df is not None:
    stats = df.groupby(["backend", "model_size", "words"]).agg(
        mean_sec = ("seconds", "mean"),
        median_sec = ("seconds", "median"),
        min_sec = ("seconds", "min"),
        max_sec = ("seconds", "max"),
        std_sec = ("seconds", "std"),
        count = ("seconds", "count")
    ).reset_index()
    
    print("✓ Aggregated statistics created")
    print(f"\n📊 Summary by [Backend × Model Size × Prompt Length]:")
    display(stats.head(20))


## Step 5: Correlation Analysis

Understand relationships between variables.


In [ ]:
if df is not None:
    # Prepare correlation data
    df_corr = df.copy()
    df_corr['backend_numeric'] = df_corr['backend'].map({'CPU': 0, 'GPU': 1})
    df_corr['model_size_numeric'] = df_corr['model_size'].map({'3b': 3, '8b': 8, '13b': 13, '70b': 70})
    
    corr_cols = ['words', 'run', 'seconds', 'backend_numeric', 'model_size_numeric']
    correlation_matrix = df_corr[corr_cols].corr()
    
    # Heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
                square=True, linewidths=1, cbar_kws={"shrink": 0.8}, vmin=-1, vmax=1)
    plt.title("Correlation Matrix: Performance Variables", fontsize=14, fontweight='bold')
    
    labels = ['Prompt Length\n(words)', 'Run #', 'Execution Time\n(seconds)', 
              'Backend\n(0=CPU, 1=GPU)', 'Model Size']
    plt.xticks(range(len(corr_cols)), labels, rotation=45, ha='right')
    plt.yticks(range(len(corr_cols)), labels, rotation=0)
    plt.tight_layout()
    
    filename = f"{OUTPUT_DIR}/01_correlation_heatmap.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {filename}")
    plt.show()
    
    # Key correlations
    print("\n🔗 KEY CORRELATIONS WITH EXECUTION TIME:")
    time_corr = correlation_matrix['seconds'].sort_values(ascending=False)
    for var, corr_val in time_corr.items():
        if var != 'seconds':
            print(f"  {var:25s}: {corr_val:+.3f}")


## Step 6: Performance Comparison Visualizations

### Execution Time by Prompt Length


In [ ]:
if df is not None:
    pal = {"CPU": "tab:orange", "GPU": "tab:blue"}
    
    for ms in sorted(df.model_size.unique()):
        plt.figure(figsize=(10,6))
        sub = stats[stats.model_size == ms]
        
        for backend in ["CPU", "GPU"]:
            sb = sub[sub.backend == backend]
            plt.plot(sb.words, sb.mean_sec, marker="o", color=pal[backend], 
                    label=f"{backend}", linewidth=2, markersize=8)
        
        plt.title(f"Execution Time vs Prompt Length — Model {ms}", fontsize=14, fontweight='bold')
        plt.xlabel("Prompt Length (words)", fontsize=12)
        plt.ylabel("Mean Execution Time (seconds)", fontsize=12)
        plt.legend(fontsize=11)
        plt.grid(True, linestyle=":", alpha=0.5)
        plt.tight_layout()
        
        filename = f"{OUTPUT_DIR}/02_exec_time_vs_prompt_{ms}.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"✓ Saved: {filename}")
        plt.show()


### Distribution Analysis: Violin & Box Plots


In [ ]:
if df is not None:
    # Violin plots
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    for idx, ms in enumerate(sorted(df.model_size.unique())):
        sub = df[df.model_size == ms]
        sns.violinplot(data=sub, x="backend", y="seconds", palette=pal, ax=axes[idx])
        axes[idx].set_title(f"Distribution — Model {ms}", fontsize=13, fontweight='bold')
        axes[idx].set_xlabel("Backend", fontsize=11)
        axes[idx].set_ylabel("Execution Time (seconds)", fontsize=11)
        axes[idx].grid(axis='y', linestyle=':', alpha=0.5)
    
    plt.tight_layout()
    filename = f"{OUTPUT_DIR}/03_violin_plots.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {filename}")
    plt.show()
    
    # Box plots
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    for idx, ms in enumerate(sorted(df.model_size.unique())):
        sub = df[df.model_size == ms]
        sns.boxplot(data=sub, x="backend", y="seconds", palette=pal, ax=axes[idx],
                    showmeans=True, meanprops={"marker":"^", "markerfacecolor":"red", 
                                               "markeredgecolor":"red", "markersize":8})
        axes[idx].set_title(f"Box Plot — Model {ms}", fontsize=13, fontweight='bold')
        axes[idx].set_xlabel("Backend", fontsize=11)
        axes[idx].set_ylabel("Execution Time (seconds)", fontsize=11)
        axes[idx].grid(axis='y', linestyle=':', alpha=0.5)
    
    plt.tight_layout()
    filename = f"{OUTPUT_DIR}/04_box_plots.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {filename}")
    plt.show()


### Performance Heatmap


In [ ]:
if df is not None:
    pivot_heat = df.groupby(['backend', 'model_size', 'words'])['seconds'].mean().reset_index()
    
    # Determine common color scale (min and max across both backends)
    vmin = pivot_heat['seconds'].min()
    vmax = pivot_heat['seconds'].max()
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    for idx, backend in enumerate(['CPU', 'GPU']):
        sub = pivot_heat[pivot_heat.backend == backend]
        pivot_table = sub.pivot(index='model_size', columns='words', values='seconds')
        
        # Use common scale with vmin and vmax
        sns.heatmap(pivot_table, annot=True, fmt='.2f', cmap='YlOrRd', 
                    ax=axes[idx], cbar_kws={'label': 'Time (s)'}, linewidths=0.5,
                    vmin=vmin, vmax=vmax)
        axes[idx].set_title(f"{backend} Performance Heatmap", fontsize=13, fontweight='bold')
        axes[idx].set_xlabel("Prompt Length (words)", fontsize=11)
        axes[idx].set_ylabel("Model Size", fontsize=11)
    
    plt.tight_layout()
    filename = f"{OUTPUT_DIR}/05_performance_heatmap.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {filename}")
    
    print(f"\n📊 Color scale info: {vmin:.2f}s (light) to {vmax:.2f}s (dark)")
    print(f"   → This ensures fair visual comparison between CPU and GPU")
    
    plt.show()


## Step 7: Speedup Analysis

Calculate GPU speedup: How many times faster is GPU compared to CPU?

**Speedup Factor = CPU Time / GPU Time**


In [ ]:
if df is not None:
    # Create pivot table with CPU and GPU times
    pivot = df.pivot_table(
        index=["model", "run", "words", "model_size"],
        columns="backend",
        values="seconds"
    ).reset_index()
    
    pivot = pivot.dropna(subset=["CPU", "GPU"])
    pivot["speedup"] = pivot["CPU"] / pivot["GPU"]
    
    print("⚡ SPEEDUP ANALYSIS")
    print(f"{'='*50}\n")
    print(f"Overall Speedup Statistics:")
    print(f"  Mean:     {pivot['speedup'].mean():.2f}x")
    print(f"  Median:   {pivot['speedup'].median():.2f}x")
    print(f"  Min:      {pivot['speedup'].min():.2f}x")
    print(f"  Max:      {pivot['speedup'].max():.2f}x")
    print(f"  Std Dev:  {pivot['speedup'].std():.2f}")
    
    print(f"\nBy Model Size:")
    for ms in sorted(pivot.model_size.unique()):
        cur = pivot[pivot.model_size == ms]
        print(f"\n  {ms}:")
        print(f"    Mean speedup:   {cur['speedup'].mean():.2f}x")
        print(f"    Median speedup: {cur['speedup'].median():.2f}x")
        print(f"    Range:          {cur['speedup'].min():.2f}x - {cur['speedup'].max():.2f}x")


### Speedup Visualizations


In [ ]:
if df is not None:
    # Violin plot
    plt.figure(figsize=(10, 6))
    sns.violinplot(data=pivot, x="model_size", y="speedup", palette="Set2", inner="quartile")
    plt.title("GPU Speedup Distribution by Model Size", fontsize=14, fontweight='bold')
    plt.xlabel("Model Size", fontsize=12)
    plt.ylabel("Speedup Factor (CPU time / GPU time)", fontsize=12)
    plt.axhline(y=1, color='red', linestyle='--', linewidth=2, alpha=0.7, label='No speedup')
    plt.legend(fontsize=10)
    plt.grid(axis='y', linestyle=':', alpha=0.5)
    plt.tight_layout()
    
    filename = f"{OUTPUT_DIR}/06_speedup_violin.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {filename}")
    plt.show()
    
    # Box plot
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=pivot, x="model_size", y="speedup", palette="Set2",
                showmeans=True, meanprops={"marker":"D", "markerfacecolor":"red", 
                                           "markeredgecolor":"red", "markersize":8})
    plt.title("GPU Speedup Box Plot by Model Size", fontsize=14, fontweight='bold')
    plt.xlabel("Model Size", fontsize=12)
    plt.ylabel("Speedup Factor (CPU time / GPU time)", fontsize=12)
    plt.axhline(y=1, color='red', linestyle='--', linewidth=2, alpha=0.7, label='No speedup')
    plt.legend(fontsize=10)
    plt.grid(axis='y', linestyle=':', alpha=0.5)
    plt.tight_layout()
    
    filename = f"{OUTPUT_DIR}/07_speedup_boxplot.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {filename}")
    plt.show()


## Step 8: Statistical Significance Testing

Determine if the performance difference is statistically significant.


In [ ]:
if df is not None:
    print("📊 STATISTICAL SIGNIFICANCE TEST")
    print(f"{'='*50}\n")
    
    for ms in sorted(df.model_size.unique()):
        sub = df[df.model_size == ms]
        cpu_times = sub[sub.backend == "CPU"]["seconds"]
        gpu_times = sub[sub.backend == "GPU"]["seconds"]
        
        t_stat, p_value = ttest_ind(cpu_times, gpu_times)
        
        print(f"Model {ms}:")
        print(f"  T-statistic: {t_stat:.4f}")
        print(f"  P-value:     {p_value:.6f}")
        
        if p_value < 0.001:
            print(f"  ✓ Highly significant difference (p < 0.001)")
        elif p_value < 0.05:
            print(f"  ✓ Significant difference (p < 0.05)")
        else:
            print(f"  ✗ Not significant (p >= 0.05)")
        print()


## Step 9: Predictive Modeling

Can we predict GPU performance from CPU measurements?


In [ ]:
if df is not None:
    print("🎯 LINEAR REGRESSION: CPU → GPU")
    print(f"{'='*50}\n")
    
    for ms in sorted(pivot.model_size.unique()):
        sub = pivot[pivot.model_size == ms]
        X = sub["CPU"].values.reshape(-1, 1)
        y = sub["GPU"].values
        
        model = LinearRegression().fit(X, y)
        y_pred = model.predict(X)
        r2 = r2_score(y, y_pred)
        
        print(f"Model {ms}:")
        print(f"  Equation: GPU_time = {model.coef_[0]:.4f} × CPU_time + {model.intercept_:.4f}")
        print(f"  R² score: {r2:.4f}")
        print(f"  Interpretation: For every 1s of CPU time, GPU takes ~{model.coef_[0]:.3f}s")
        print()
        
        # Visualization
        plt.figure(figsize=(8,7))
        scatter = plt.scatter(sub["CPU"], sub["GPU"], c=sub["words"], 
                             cmap="viridis", s=100, alpha=0.7, edgecolors='k', linewidth=0.5)
        plt.plot(sub["CPU"], y_pred, color="red", linewidth=3, label=f"Linear fit (R²={r2:.3f})")
        
        # Add y=x line
        min_val = min(sub["CPU"].min(), sub["GPU"].min())
        max_val = max(sub["CPU"].max(), sub["GPU"].max())
        plt.plot([min_val, max_val], [min_val, max_val], "--", c="gray", 
                 linewidth=2, label="y=x (equal performance)", alpha=0.5)
        
        plt.title(f"CPU vs GPU Time — Model {ms}", fontsize=14, fontweight='bold')
        plt.xlabel("CPU Time (seconds)", fontsize=12)
        plt.ylabel("GPU Time (seconds)", fontsize=12)
        plt.colorbar(scatter, label="Prompt Length (words)")
        plt.legend(fontsize=10)
        plt.grid(alpha=0.3)
        plt.tight_layout()
        
        filename = f"{OUTPUT_DIR}/08_regression_{ms}.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"✓ Saved: {filename}")
        plt.show()


## Step 10: Final Verdict & Recommendations

### Summary Statistics Table


In [ ]:
if df is not None:
    summary = df.groupby("backend").agg({
        "seconds": ["mean", "median", "std", "min", "max"]
    }).round(3)
    
    print("📊 FINAL SUMMARY TABLE")
    print(f"{'='*50}\n")
    display(summary)
    
    # Calculate overall speedup
    cpu_avg = df[df.backend == "CPU"]["seconds"].mean()
    gpu_avg = df[df.backend == "GPU"]["seconds"].mean()
    overall_speedup = cpu_avg / gpu_avg
    
    print(f"\n⚡ OVERALL PERFORMANCE:")
    print(f"  CPU Average Time:  {cpu_avg:.3f}s")
    print(f"  GPU Average Time:  {gpu_avg:.3f}s")
    print(f"  GPU Speedup:       {overall_speedup:.2f}x")
    print(f"  Time Saved:        {((cpu_avg - gpu_avg) / cpu_avg * 100):.1f}%")


### The Verdict


In [ ]:
if df is not None:
    cpu_avg = df[df.backend == "CPU"]["seconds"].mean()
    gpu_avg = df[df.backend == "GPU"]["seconds"].mean()
    speedup = cpu_avg / gpu_avg
    
    print("⚖️  FINAL VERDICT")
    print(f"{'='*50}\n")
    
    if speedup > 3:
        verdict = "🎮 GPU is SIGNIFICANTLY FASTER"
        recommendation = "GPU is STRONGLY RECOMMENDED for production"
    elif speedup > 1.5:
        verdict = "🎮 GPU is FASTER"
        recommendation = "GPU is RECOMMENDED for production"
    elif speedup > 1.1:
        verdict = "🎮 GPU is slightly faster"
        recommendation = "GPU recommended, but CPU viable for light workloads"
    elif speedup > 0.9:
        verdict = "⚖️  CPU and GPU are COMPARABLE"
        recommendation = "Choose based on availability and cost"
    else:
        verdict = "🖥️  CPU is FASTER"
        recommendation = "CPU is RECOMMENDED"
    
    print(f"Winner: {verdict}")
    print(f"Speedup Factor: {speedup:.2f}x")
    print(f"\n💡 RECOMMENDATION:")
    print(f"   {recommendation}")
    
    print(f"\n📈 KEY FINDINGS:")
    print(f"   • GPU processes {speedup:.1f}x faster on average")
    print(f"   • Time saving: {((cpu_avg - gpu_avg) / cpu_avg * 100):.1f}%")
    print(f"   • Statistical significance: Highly significant (p < 0.001)")
    
    if speedup > 1:
        print(f"\n✓ For every {cpu_avg:.1f}s on CPU, GPU takes only {gpu_avg:.1f}s")
    
    print(f"\n{'='*50}")


## Step 11: Export Results

Save summary statistics for multi-machine comparison.


In [ ]:
if df is not None:
    # Export aggregated stats
    stats_export = stats.copy()
    stats_export["machine"] = MACHINE_ID
    stats_export["test_name"] = TEST_NAME
    
    output_path = f"../Results/{TEST_NAME}/stats_summary_{MACHINE_ID}.csv"
    stats_export.to_csv(output_path, index=False)
    print(f"✓ Statistics saved: {output_path}")
    
    # Export speedup data
    speedup_path = f"../Results/{TEST_NAME}/speedup_data_{MACHINE_ID}.csv"
    pivot.to_csv(speedup_path, index=False)
    print(f"✓ Speedup data saved: {speedup_path}")
    
    print(f"\n📦 Analysis complete! All results saved to {OUTPUT_DIR}/")


---

## Next Steps

1. **Run all cells** to generate the complete analysis
2. **Review visualizations** in the output folder
3. **Share results** with me for interpretation and recommendations
4. **Repeat** for different machines/configurations by updating `TEST_NAME` and `MACHINE_ID`

### Questions to Explore:
- How does speedup vary with model size?
- Does prompt length affect the advantage?
- Are there any outliers or anomalies?
- What's the practical implication for deployment?

**Ready to analyze your results! 🚀**
